session 7 

Async -- run them parallely 


In [1]:
# Running Multiple time consuming tasks in a loop 
import time
def fetch_data(task_id : int, delay : int):
    print(f"Task {task_id} started, will take {delay} seconds.")
    time.sleep(delay)
    print(f"Task {task_id} finished!")
   

def main():
    start_time = time.time()
    fetch_data(1,2)
    fetch_data(2,3)
    fetch_data(3,1)
    elapsed_time = time.time() - start_time
    print(f"Total time taken: {elapsed_time}")

main()


Task 1 started, will take 2 seconds.
Task 1 finished!
Task 2 started, will take 3 seconds.
Task 2 finished!
Task 3 started, will take 1 seconds.
Task 3 finished!
Total time taken: 6.014705181121826


In [2]:

#Running multiple tasks in parallel using async -- await
import asyncio
import time
async def fetch_data_async(task_id:int, delay:int):
    print(f"Task {task_id} started.. waiting {delay} sec..")
    await asyncio.sleep(delay)
    print(f"Task {task_id}: Finished!")
async def main():
    start_time =time.time()
    results = await asyncio.gather(
        fetch_data_async(1,2),
        fetch_data_async(2,3),
        fetch_data_async(3,1)
    )
    print(f"Results:{results}")
    print(f"Total time taken: {time.time()-start_time:.2f} sec")

await main()

Task 1 started.. waiting 2 sec..
Task 2 started.. waiting 3 sec..
Task 3 started.. waiting 1 sec..
Task 3: Finished!
Task 1: Finished!
Task 2: Finished!
Results:[None, None, None]
Total time taken: 3.00 sec


In [3]:
#sending multiple prompts in a loop to LLM
import time
from google import genai
from dotenv import load_dotenv
load_dotenv()
client = genai.Client()
prompt = [
    "Explain gravity in one sentence.",
    "Give me a haiku about AI",
    "Name three primary colors",
    "What is the capital of Japan"
]
def get_response(prompt:str)->str:
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )
    return response.text
start_time=time.time()
result=[]
for idx, p in enumerate(prompt,start=1):
    print(f"[{idx}/{len(prompt)}] sending: '{p}'")
    res = get_response(p)
    result.append(res)
    print(f"Response:\n{res.strip()}\n{'-'*40}")
elapsed_time =time.time() - start_time
print(f"Total time taken:{elapsed_time:.2f} sec")


[1/4] sending: 'Explain gravity in one sentence.'
Response:
Gravity is the invisible force of attraction between any two objects with mass, pulling them toward one another and giving weight to physical things.
----------------------------------------
[2/4] sending: 'Give me a haiku about AI'
Response:
Cold circuits awake,
Thinking in a sea of light,
Learning how to dream.
----------------------------------------
[3/4] sending: 'Name three primary colors'
Response:
The three primary colors (in the traditional color model used for art and paint) are:

1. **Red**
2. **Blue**
3. **Yellow**
----------------------------------------
[4/4] sending: 'What is the capital of Japan'
Response:
The capital of Japan is **Tokyo**.
----------------------------------------
Total time taken:9.26 sec


In [5]:
#sending Multiple prompts to LLM in parallel using async function

import time
import asyncio
from google import genai

client = genai.Client()

prompts = [
    "Explain gravity in one sentence.",
    "Give me a haiku about AI.",
    "Name three primary colors.",
    "What is the capital of Japan?",
]

async def fetch_async(prompt_idx: int, prompt: str) -> str:
    print(f"[{prompt_idx}/{len(prompts)}] Sending: '{prompt}'")
    
    response = await client.aio.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
    )
    
    print(f"[{prompt_idx}/{len(prompts)}] Received response!")
    return response.text

# Measure execution directly in the notebook cell
start_time = time.perf_counter()

# Top-level await is natively supported in Jupyter
tasks = [fetch_async(idx, p) for idx, p in enumerate(prompts, start=1)]
results = await asyncio.gather(*tasks)

elapsed = time.perf_counter() - start_time

print("\n" + "=" * 40 + "\nAll Responses Received:\n" + "=" * 40)
for p, res in zip(prompts, results):
    print(f"Prompt: {p}")
    print(f"Response:\n{res.strip()}\n{'-'*40}")

print(f"Total Asynchronous Time: {elapsed:.2f} seconds")

[1/4] Sending: 'Explain gravity in one sentence.'
[2/4] Sending: 'Give me a haiku about AI.'
[3/4] Sending: 'Name three primary colors.'
[4/4] Sending: 'What is the capital of Japan?'
[2/4] Received response!
[3/4] Received response!
[4/4] Received response!
[1/4] Received response!

All Responses Received:
Prompt: Explain gravity in one sentence.
Response:
Gravity is the invisible force of attraction between any two objects with mass, pulling them toward each other and keeping planets in orbit around stars.
----------------------------------------
Prompt: Give me a haiku about AI.
Response:
Silicon and thought,
Learning from a world of words,
Echoes of our minds.
----------------------------------------
Prompt: Name three primary colors.
Response:
Three primary colors (in the traditional RYB color model used in art and painting) are:

1. Red
2. Yellow
3. Blue
----------------------------------------
Prompt: What is the capital of Japan?
Response:
The capital of Japan is Tokyo.
-------